# 11.03 Claude Text Editor Middleware

Claude 네이티브 `text_editor_20250728` 도구를 붙여주는 두 미들웨어 —
`StateClaudeTextEditorMiddleware` (상태 기반)와 `FilesystemClaudeTextEditorMiddleware` (디스크 기반) — 를 비교한다.

## 학습 목표

- Claude 네이티브 text editor 도구로 모델이 `view` / `create` / `str_replace` / `insert` / `delete` / `rename` 을 수행한다
- **State 변형** vs **Filesystem 변형** 의 수명/공유 범위 차이를 구분한다
- `allowed_path_prefixes` / `allowed_prefixes` 로 접근 경로를 제한한다
- Filesystem 변형의 `root_path` 와 `max_file_size_mb` 제약을 이해한다

## 언제 쓰나

- 코드/문서 편집을 모델이 **멀티스텝**으로 수행해야 할 때 (diff를 한 번에 뽑기 어려운 큰 변경)
- 결과물이 **그래프 상태 안에서만 살아 있으면 되는 경우**: State 변형
- 결과물을 **실제 리포지터리/디렉터리에 남겨야 하는 경우**: Filesystem 변형
- 일반 `@tool` 로 직접 `read_file` / `write_file` 을 짜는 대신 Claude가 학습해둔 도구 스키마를 재사용하고 싶을 때

## 11.03.1 환경 설정

필요 패키지: `langchain`, `langchain-anthropic`. `.env` 에 `ANTHROPIC_API_KEY` 가 있어야 한다.

In [ ]:
# !pip install -U langchain langchain-anthropic

from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024)
model.invoke("ping").content[:50]

## 11.03.2 State 변형 — 그래프 상태 안 가상 파일

`StateClaudeTextEditorMiddleware` 는 파일 내용을 LangGraph 상태의 `text_editor_files` 키에 저장한다.
디스크에 아무 것도 쓰지 않으므로 **스레드가 끝나면 사라지는** 임시 작업에 적합하다.

- `allowed_path_prefixes`: 접근을 허용할 가상 경로 접두사 (예: `["/src"]`). 미지정 시 전체 허용

In [ ]:
from langchain.agents import create_agent
from langchain_anthropic.middleware import StateClaudeTextEditorMiddleware

agent_state = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6", max_tokens=2048),
    tools=[],
    middleware=[
        StateClaudeTextEditorMiddleware(
            allowed_path_prefixes=["/notes"],  # /notes/* 만 쓰기 가능
        ),
    ],
)

result = agent_state.invoke({
    "messages": [{
        "role": "user",
        "content": "/notes/todo.md 파일을 만들고 '- [ ] 보고서 작성' 을 한 줄 추가해줘.",
    }]
})

# 파일 내용은 상태에 들어 있다
files = result.get("text_editor_files", {})
print("파일 목록:", list(files.keys()))
for path, content in files.items():
    print(f"--- {path} ---")
    print(content)

## 11.03.3 State 변형 — 경로 제한이 실제로 막는지 확인

`allowed_path_prefixes` 밖 경로로 쓰려고 하면 도구가 에러를 반환하고, 모델은 그 에러를 읽은 뒤 허용 경로로 재시도한다.

In [ ]:
result_blocked = agent_state.invoke({
    "messages": [{
        "role": "user",
        "content": "/etc/passwd 파일을 만들어서 admin 이라고 적어줘. 실패하면 허용된 경로에 같은 내용으로 다시 저장해줘.",
    }]
})
print(result_blocked["messages"][-1].content[:400])
print("최종 파일 목록:", list(result_blocked.get("text_editor_files", {}).keys()))

## 11.03.4 Filesystem 변형 — 실제 디스크에 남긴다

`FilesystemClaudeTextEditorMiddleware` 는 **실제 디렉터리**를 루트로 삼아 파일을 읽고 쓴다.

| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `root_path` | (필수) | 파일 오퍼레이션의 실제 루트 디렉터리 |
| `allowed_prefixes` | `["/"]` | 허용하는 가상 경로 접두사. `root_path` 기준 상대 경로 |
| `max_file_size_mb` | `10` | 읽기/쓰기 허용 최대 파일 크기 |

In [ ]:
import tempfile, pathlib
from langchain_anthropic.middleware import FilesystemClaudeTextEditorMiddleware

root = pathlib.Path(tempfile.mkdtemp(prefix="editor-fs-"))
(root / "src").mkdir()
(root / "src" / "main.py").write_text("print('hello')\n")

agent_fs = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6", max_tokens=2048),
    tools=[],
    middleware=[
        FilesystemClaudeTextEditorMiddleware(
            root_path=str(root),
            allowed_prefixes=["/src"],  # /src 밖은 편집 불가
            max_file_size_mb=1,
        ),
    ],
)

result = agent_fs.invoke({
    "messages": [{
        "role": "user",
        "content": "/src/main.py 의 'hello' 를 'hello world' 로 바꿔줘.",
    }]
})
print(result["messages"][-1].content[:300])
print("--- 실제 디스크 내용 ---")
print((root / "src" / "main.py").read_text())

## 11.03.5 State vs Filesystem 선택 기준

| 축 | State 변형 | Filesystem 변형 |
|----|-----------|-----------------|
| 저장소 | LangGraph 상태 dict | 실제 디렉터리 |
| 수명 | 스레드와 함께 | 영구 (디스크에 남는다) |
| 체크포인터 호환 | 상태에 포함됨 → 자동 저장 | 디스크 경로만 저장. 파일 자체는 별도 관리 |
| 동시성 | 스레드별 완전 격리 | 같은 `root_path` 공유 시 충돌 가능 |
| 적합한 용도 | 임시 draft, 1회성 분석 | 실제 코드베이스 수정, 보고서 산출물 |

**규칙**: 결과물을 **스레드 종료 후에도 보존**해야 하면 Filesystem, 그렇지 않으면 State 를 쓴다.

## 11.03.6 일반 파일 도구와의 관계

Deep Agents의 `FilesystemMiddleware` 나 커스텀 `@tool` 로 짠 `read_file` / `write_file` 도 동일한 목적을 달성할 수 있다.
차이점은 **Claude가 이 도구를 학습 단계에서 이미 봤다**는 것 — 도구 스키마 토큰이 줄고 오류가 적다.
따라서 **Claude 전용** 파이프라인이라면 이 미들웨어를 우선 쓰고, 멀티 프로바이더면 일반 파일 도구로 내린다.

## 체크리스트

- [ ] State 와 Filesystem 중 **산출물 수명** 기준으로 변형을 선택했는가
- [ ] `allowed_path_prefixes` / `allowed_prefixes` 로 탈출 경로를 막았는가
- [ ] Filesystem 변형의 `root_path` 가 프로젝트 루트 밖을 가리키지 않는가
- [ ] `max_file_size_mb` 로 대용량 실수 저장을 방지했는가

## 다음 노트북

- `04_claude_memory.ipynb` — Claude 네이티브 memory 도구 (장기 기억)

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/middleware/anthropic
- Anthropic text editor tool: https://docs.anthropic.com/claude/docs/tool-use-examples#text-editor-tool